# Rover VLA — training on Colab

Trains the rover waypoint policy on a Colab GPU. Pick a **MODE** below.

| mode | what it does | needs |
|---|---|---|
| `A` | **stage-1 baseline** — frozen VLM backbone, action expert only. The stable recipe. | dataset |
| `B` | **constrained vision adaptation** — warm-start a checkpoint, unfreeze only the top-K vision layers (LM frozen). Tests whether adapting vision fixes colour grounding. | dataset + checkpoint |
| `C` | **deeper LM** — raise `num_vlm_layers` (SmolVLA truncates the LM to 16 of 32 by default). Tests whether the truncation is what blocks fine colour binding. Trains from base. | dataset |

Pipeline: **data (local Gazebo) → train (Colab) → eval (local sim + policy_server)**.
Colab does training only.

Upload from `rover/colab_upload/` to Drive `MyDrive/rover_vla/` first:
`rover_vla_v3.tar.gz` (always) and `stage1_v3_pretrained.tar.gz` (mode B only).

## 1. Config — set this, then Runtime ▸ Run all

`SMOKE = True` runs a ~30-step check on whatever GPU you're on (a T4 is fine) to
confirm data load, model build, the train loop, and checkpoint-save all work —
same code paths as the real run, just tiny. Flip to `False` for the real run
(A100 for mode C).

In [ ]:
MODE = 'C'          # 'A' baseline | 'B' vision adaptation | 'C' deeper LM
SMOKE = False       # tiny end-to-end run on the current GPU (T4) to check the
                    # pipeline before the real full run on A100. Set False for real.

STEPS = 10000
SAVE_FREQ = 2000    # checkpoints go to local Colab disk; only the last is copied to Drive
VISION_TOPK = 2     # mode B: how many top vision layers to unfreeze
NUM_VLM_LAYERS = 32 # mode C: LM layers to keep (SmolVLA default 16 of 32)

if SMOKE:
    STEPS = 30      # a few log lines + cross one save
    SAVE_FREQ = 20  # exercise the checkpoint-save path within the run

DRIVE = '/content/drive/MyDrive/rover_vla'
DATA_TAR = f'{DRIVE}/rover_vla_v3.tar.gz'
CKPT_TAR = f'{DRIVE}/stage1_v3_pretrained.tar.gz'   # mode B
RUN_NAME = {'A': 'stage1_colab', 'B': 'stage1c_colab', 'C': 'stage1d_deeplm'}[MODE]
if SMOKE:
    RUN_NAME += '_smoke'
print('mode', MODE, ('SMOKE' if SMOKE else 'FULL'), '->', RUN_NAME, '| steps', STEPS)

mode C FULL -> stage1d_deeplm | steps 10000


## 2. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0


## 3. Install

In [ ]:
# [smolvla] = the policy; [dataset] pulls in PyAV (`av`) for --video_backend=pyav
!pip install -q "lerobot[smolvla,dataset]" wandb
import lerobot, torch, av
print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| av', av.__version__)

lerobot 0.6.0 | torch 2.11.0+cu128 | av 15.1.0


## 4. Data (+ checkpoint for mode B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, tarfile
os.makedirs('/content/data', exist_ok=True)
with tarfile.open(DATA_TAR) as t:
    t.extractall('/content/data')
DATASET_ROOT = '/content/data/rover_vla_v3'
print('dataset:', DATASET_ROOT, os.path.isdir(DATASET_ROOT))

POLICY_PATH = 'lerobot/smolvla_base'
if MODE == 'B':
    os.makedirs('/content/ckpt', exist_ok=True)
    with tarfile.open(CKPT_TAR) as t:
        t.extractall('/content/ckpt')
    POLICY_PATH = '/content/ckpt/pretrained_model'
    print('warm-start from:', POLICY_PATH, os.path.isdir(POLICY_PATH))
print('policy path:', POLICY_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_1383/1803008995.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall('/content/data')


dataset: /content/data/rover_vla_v3 True
policy path: lerobot/smolvla_base


## 5. Weights & Biases
Put your key in a Colab **secret** named `WANDB_API_KEY` (key icon, left bar).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    import wandb; wandb.login()
print('wandb key set:', bool(os.environ.get('WANDB_API_KEY')))

wandb key set: False


## 6. Precision + mode-B patch
bf16 needs compute capability >= 8.0 (A100/L4). T4/V100 fall back to fp32.
Mode B also patches the freeze logic to unfreeze only the top-K vision layers.

In [ ]:
import torch, pathlib
cc = torch.cuda.get_device_capability()
bf16_ok = cc[0] >= 8
print('compute capability', cc, '| bf16 native:', bf16_ok)

import lerobot
f = pathlib.Path(lerobot.__file__).parent / 'policies/smolvla/smolvlm_with_expert.py'
s = f.read_text()
if not bf16_ok:
    s = s.replace('torch_dtype="bfloat16"', 'torch_dtype="float32"')
    print('patched VLM load -> float32')

if MODE == 'B':
    old = '        else:\n            # To avoid unused params issue with distributed training'
    new = ('        else:\n'
           '            for _p in self.get_vlm_model().text_model.parameters():\n'
           '                _p.requires_grad = False\n'
           '            _vm = self.get_vlm_model().vision_model\n'
           '            for _p in _vm.parameters():\n'
           '                _p.requires_grad = False\n'
           f'            for _l in _vm.encoder.layers[-{VISION_TOPK}:]:\n'
           '                for _p in _l.parameters():\n'
           '                    _p.requires_grad = True\n'
           '            # To avoid unused params issue with distributed training')
    assert old in s, 'freeze-branch anchor not found'
    s = s.replace(old, new)
    print(f'patched: top-{VISION_TOPK} vision unfreeze, LM frozen')
f.write_text(s)

gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 64 if gb > 30 else (32 if gb > 20 else 16)
if MODE == 'C':
    BATCH = max(8, BATCH // 2)   # deeper LM costs memory
if SMOKE:
    BATCH = min(BATCH, 8)        # keep the T4 smoke run light and fast
print('VRAM %.0f GB -> batch %d%s' % (gb, BATCH, '  (smoke)' if SMOKE else ''))

compute capability (8, 0) | bf16 native: True
VRAM 42 GB -> batch 32


## 7. Train
Action space: flat 10x(x,y,v)=30-dim chunk with `chunk_size=1` (lerobot's
delta-timestamps chunking would mix body frames).

In [ ]:
extra = ''
if MODE == 'B':
    extra = '--policy.train_expert_only=false --policy.freeze_vision_encoder=false'
elif MODE == 'C':
    extra = f'--policy.num_vlm_layers={NUM_VLM_LAYERS}'

# smoke runs stay offline: no wandb key needed, nothing logged to the project
if SMOKE:
    wandb_flags = '--wandb.enable=false'
    LOG_FREQ = 5
else:
    wandb_flags = ("--wandb.enable=true --wandb.project=rover-vla --wandb.mode=online "
                   f"--wandb.notes='{RUN_NAME} (colab, mode {MODE})'")
    LOG_FREQ = 100


OUTPUT_DIR = f'/content/outputs/{RUN_NAME}'
!rm -rf {OUTPUT_DIR}

cmd = f'''lerobot-train \
  --policy.path={POLICY_PATH} --policy.push_to_hub=false \
  --policy.chunk_size=1 --policy.n_action_steps=1 {extra} \
  --dataset.repo_id=local/rover_vla_v3 --dataset.root={DATASET_ROOT} \
  --dataset.video_backend=pyav \
  --rename_map='{{"observation.image": "observation.images.camera1"}}' \
  --batch_size={BATCH} --steps={STEPS} --save_freq={SAVE_FREQ} --log_freq={LOG_FREQ} \
  {wandb_flags} \
  --output_dir={OUTPUT_DIR}'''
print(cmd)
!{cmd}

lerobot-train   --policy.path=lerobot/smolvla_base --policy.push_to_hub=false   --policy.chunk_size=1 --policy.n_action_steps=1 --policy.num_vlm_layers=32   --dataset.repo_id=local/rover_vla_v3 --dataset.root=/content/data/rover_vla_v3   --dataset.video_backend=pyav   --rename_map='{"observation.image": "observation.images.camera1"}'   --batch_size=32 --steps=10000 --save_freq=2000 --log_freq=100   --wandb.enable=true --wandb.project=rover-vla --wandb.mode=online --wandb.notes='stage1d_deeplm (colab, mode C)'   --output_dir=/content/outputs/stage1d_deeplm
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-07-21 10:28:46 ot_train.py:232 {'batch_size': 32,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'depth_o

## 7b. Probe — is the deeper LM real? (mode C)
The `Missing key(s) … layers.16–31` warning at load time is expected: `smolvla_base`
only shipped 16 VLM layers. What matters for mode C is whether layers 16–31 hold the
**original SmolVLM2 pretrained** weights (frozen) or were left default-initialised.
This compares a frozen deep VLM layer in the trained checkpoint against the cached
SmolVLM2 backbone — `matches SmolVLM2 pretrained: True` means mode C is genuinely
testing a deeper *pretrained* LM.</cell id="cell-14">

In [ ]:
import glob, torch
from safetensors import safe_open

def _get(pattern, suffix):
    for p in glob.glob(pattern, recursive=True):
        try:
            with safe_open(p, framework='pt') as fh:
                for k in fh.keys():
                    if k.endswith(suffix):
                        return fh.get_tensor(k), p
        except Exception:
            pass
    return None, None

sfx = 'text_model.layers.31.self_attn.q_proj.weight'
ckpt, _ = _get(f'{OUTPUT_DIR}/checkpoints/last/pretrained_model/*.safetensors', sfx)
# cache dir is models--HuggingFaceTB--SmolVLM2-500M-... so the glob needs a leading *
base, bpath = _get('/root/.cache/huggingface/**/*SmolVLM2*/**/*.safetensors', sfx)
if ckpt is None or base is None:
    print('probe: layer-31 tensor not found (ckpt found:', ckpt is not None,
          '| base found:', base is not None, ')')
else:
    same = torch.allclose(ckpt.float(), base.float(), atol=1e-4)
    print('base:', bpath)
    print(f'VLM layer-31 q_proj  ckpt_std={ckpt.std():.4f}  smolvlm2_std={base.std():.4f}')
    print('matches SmolVLM2 pretrained (deeper LM is real, frozen):', same)

base file: /root/.cache/huggingface/hub/models--HuggingFaceTB--SmolVLM2-500M-Video-Instruct/snapshots/7b375e1b73b11138ff12fe22c8f2822d8fe03467/model.safetensors
ckpt layer-31 q_proj std = 0.1133
matches SmolVLM2 pretrained (deeper LM is real, frozen): True


## 8. Save the checkpoint back to Drive
Then download it locally and evaluate with `rover/runtime/policy_server.py`
+ `ros2 run rover_expert run_eval.py --swap`.

On a `SMOKE` run this still exercises the tar/save path, but the checkpoint is
under `checkpoints/<run>_smoke.tar.gz` and is throwaway — don't evaluate it.

In [ ]:
import tarfile, os
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
OUT = f'{DRIVE}/checkpoints/{RUN_NAME}.tar.gz'
with tarfile.open(OUT, 'w:gz') as t:
    t.add(f'{OUTPUT_DIR}/checkpoints/last/pretrained_model', arcname='pretrained_model')
print('saved', OUT, os.path.getsize(OUT) // 1024 // 1024, 'MB')

saved /content/drive/MyDrive/rover_vla/checkpoints/stage1d_deeplm.tar.gz 1077 MB
